# 🔥 Mojo/MAX M3 real decoder block on Colab T4 (issue #57)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vedmalex/higgs-local-test/blob/main/notebooks/mojo_max_m3_t4.ipynb)

Re-runs, **unchanged in logic**, on a real Colab Tesla T4 the five M3 tasks that were only
validated on M1/Metal so far (per [`docs/research/mojo-max/m3-plan.md`](../docs/research/mojo-max/m3-plan.md)'s
M3-10 — "Stage E — Tesla T4 re-validation"):

1. **M3-1** — mixed CPU/GPU placement spike (`m3_device_mixing_spike.py`): does a CPU-placed
   `ops.conv2d_transpose` inside an otherwise-GPU MAX graph dispatch cleanly under real CUDA,
   the way it does under Metal?
2. **M3-5** — the full `_BosonDecoderBlock` (`stride=5`, synthetic weights) as one mixed
   CPU/GPU MAX graph, across the same 6 seeds M1 used.
3. **M3-6** — the same graph against the **real** stride-5 block-1 weights extracted from
   `bosonai/higgs-tts-3-4b`, seed 99 (M1's convention).
4. **M3-7** — the `stride=8` shape-coverage case, synthetic weights, the same 6 seeds M1 used.
5. **M3-9** — the BF16-storage precision pass (`fp32` / `bf16-cast` / `bf16-nocast`) over the
   real stride-5 block, seed 99.

`_BosonResidualUnit`'s own Tesla T4 GPU gap (`m2-residual-unit-results.md`) is closed for free
here since M3-5 exercises that composite as part of the full block.

This is a **build-only** notebook: it has not been executed on a real T4 by the agent that
wrote it. A human runs it; the raw output that comes back gets committed as
`docs/research/mojo-max/m3-block-output-t4.txt` (+ a device-mixing-specific file for M3-1), and
only THEN does `m3-plan.md`'s M3-10 checkbox get ticked and `m3-block-results.md` get its T4
confirm/contradict section, per this repo's "a box is ticked only after the thing has actually
been RUN" policy.

## Two environments, like the M1 runs that produced these scripts

Everything MAX/Mojo-related (`m3_device_mixing_spike.py`, `m3_decoder_block_prototype.py`) runs
under **pixi** (no `torch` in that env — it is stubbed at import time only). M3-6/M3-9's
`--real-weights` path needs the real checkpoint's BF16 tensors, which `safetensors`'
`framework="numpy"` cannot decode — that step needs **`torch`+`transformers`+`safetensors`**,
which on M1 lived in this repo's separate `.venv-tts`. This notebook recreates that same split:
a `.venv-tts` venv is built once (Stage 5 below) purely to run
`m3_real_weights_export.py`, which writes a plain-FP32 `.npz` cache that the pixi/MAX env then
reads with NumPy alone — `m3_decoder_block_prototype.py`'s own code is untouched, exactly as
M3-6 designed it.

Minimal blocker: only `pixi`+`modular` and, for Stage 5 only, a second throwaway venv get
installed — no TTS/STT/Qwen stack, matching the M0/M2 T4 notebooks' convention.

## Автоотключение и сохранность данных

Тот же паттерн, что в `mojo_max_m2_t4.ipynb`: ВМ отключается по завершении безусловно (квота
Colab не должна гореть на простое), поэтому каждый результат пишется на Google Drive сразу по
готовности, а не только в конце — обрыв на середине прогона не теряет то, что уже готово.


## 1. GPU and driver

Record the exact T4 driver/CUDA/compute-capability combination this run used — matching
`m3_device_mixing_spike.py`'s own toolchain-pinning discipline (M3-1's results doc records
MAX/Mojo/macOS/Xcode versions precisely; do the CUDA-side equivalent here).

In [ ]:
!nvidia-smi


In [ ]:
import subprocess

smi = subprocess.run(["nvidia-smi", "--query-gpu=driver_version,name,compute_cap",
                       "--format=csv,noheader"], capture_output=True, text=True)
print(smi.stdout.strip() or smi.stderr)


## 2. Google Drive workspace

`USE_DRIVE = False` переключает на эфемерный `/content` (например, для отладки самого
ноутбука), но по умолчанию всё пишется на Диск сразу, как только готово.


In [ ]:
USE_DRIVE = True

from pathlib import Path

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    WORKSPACE = Path("/content/drive/MyDrive/higgs-benchmark/mojo-max-m3")
else:
    WORKSPACE = Path("/content/mojo-max-m3")

OUTPUT_DIR = WORKSPACE / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Workspace: {WORKSPACE}")
print(f"Results will be written to: {OUTPUT_DIR}")


## 3. Install pixi + Mojo/MAX (stable channel 26.5)

Same convention as `mojo_max_m0_t4.ipynb` / `mojo_max_m2_t4.ipynb` — no version is hard-pinned
in the `pixi add` call itself (matches what M2's notebook did), but the actually-resolved
`mojo`/`max` versions are printed right after install so this run's toolchain is on record,
the same way M3-1's results doc records MAX 26.5.0 / Mojo 1.0.0 for the M1 run.

In [ ]:
!curl -fsSL https://pixi.sh/install.sh | bash
import os
os.environ["PATH"] = f"{os.path.expanduser('~/.pixi/bin')}:{os.environ['PATH']}"
!pixi --version


In [ ]:
!mkdir -p /content/mojo-probe-t4
!cd /content/mojo-probe-t4 && pixi init . -c https://conda.modular.com/max/ -c conda-forge
!cd /content/mojo-probe-t4 && pixi add modular
!cd /content/mojo-probe-t4 && pixi run python -c "import max; print('max import ok')"
!cd /content/mojo-probe-t4 && pixi run mojo --version
!cd /content/mojo-probe-t4 && pixi run max --version


## 4. Fetch the M3 scripts from the repo


In [ ]:
REPO_URL = "https://github.com/vedmalex/higgs-local-test.git"
REPO_REF = "main"

!rm -rf /content/higgs-local-test
!git clone --quiet --depth 1 --branch "$REPO_REF" "$REPO_URL" /content/higgs-local-test

import os
REPO_SCRIPTS_DIR = "/content/higgs-local-test/docs/research/mojo-max"
assert os.path.isdir(REPO_SCRIPTS_DIR), (
    f"Clone failed or landed somewhere else -- {REPO_SCRIPTS_DIR} does not exist. "
    "Re-run this cell before continuing; do not proceed to the next cells if this assert fires."
)
!ls $REPO_SCRIPTS_DIR/*.py
print("Repo present at", REPO_SCRIPTS_DIR)


## 0b. Diagnostic: channel-count sweep for the `ldmatrix` T4 abort

A real run of this notebook's M3-5 stage hit a fatal `LLVM ERROR: Cannot select: intrinsic
%llvm.nvvm.ldmatrix.sync.aligned.m8n8.x4.b16` abort on T4 -- distinct from the already-known
ConvTranspose1d/cuDNN bug (see issue #57). Leading hypothesis: MAX's `ops.conv2d` kernel
selection switches to a tensor-core path above some channel-count threshold, and that path has
a Turing (`sm_75`) codegen gap (confirmed as a known, general MAX/T4 issue via the Modular forum:
https://forum.modular.com/t/having-issues-with-max-matmul-on-default-google-colab-gpu-t4/1658).
M2's isolated prototypes used only 32 channels and never crossed whatever the threshold is; the
real block uses 512/256 (stride=5) and 1024/512 (stride=8).

This cell sweeps 32/64/128/256/512/1024 channels, each in its own subprocess (the abort is
fatal, not catchable), to find exactly where it starts. Run this BEFORE the M3-5+ cells below --
if it confirms the threshold, the remaining full-block stages are expected to abort identically
regardless of seed, and there is no need to let them all run to find that out empirically again.


In [ ]:
sweep_out = subprocess.run(
    ["pixi", "run", "python", "-u",
     "/content/higgs-local-test/docs/research/mojo-max/m3_ldmatrix_channel_sweep.py"],
    cwd="/content/mojo-probe-t4",
    capture_output=True,
    text=True,
)
print(sweep_out.stdout)
if sweep_out.stderr:
    print("--- stderr ---")
    print(sweep_out.stderr)

(OUTPUT_DIR / "m3-ldmatrix-channel-sweep-output-t4.txt").write_text(
    sweep_out.stdout + ("\n--- stderr ---\n" + sweep_out.stderr if sweep_out.stderr else "")
)
print(f"\nWritten to {OUTPUT_DIR / 'm3-ldmatrix-channel-sweep-output-t4.txt'}")


## 5. Stage A — M3-1: mixed CPU/GPU device-mixing spike

`m3_device_mixing_spike.py` isolates its own graph-build-and-execute step in a subprocess it
spawns itself (a Metal `cudnnCreate` GPU-dispatch bug on M1 was a fatal, uncatchable process
abort — the script's own `main()` already guards against that on any platform, T4 included),
so this cell just invokes it directly; no extra subprocess wrapper is needed here.

**What this answers on T4**: does the same CPU-placed `ops.conv2d_transpose` inside an
otherwise-GPU MAX graph dispatch to the CPU kernel cleanly under real CUDA, the way it does
under Metal (M3-1's confirmed M1 result)?


In [ ]:
import os
if not os.path.isdir("/content/higgs-local-test/docs/research/mojo-max"):
    print("/content/higgs-local-test missing -- re-cloning (a Colab runtime hiccup likely wiped /content)")
    !rm -rf /content/higgs-local-test
    !git clone --quiet --depth 1 --branch main https://github.com/vedmalex/higgs-local-test.git /content/higgs-local-test
    assert os.path.isdir("/content/higgs-local-test/docs/research/mojo-max"), "re-clone still failed -- check network/repo access"

device_mixing_output = OUTPUT_DIR / "m3-device-mixing-output-t4.txt"
!cd /content/mojo-probe-t4 && pixi run python -u /content/higgs-local-test/docs/research/mojo-max/m3_device_mixing_spike.py 2>&1 | tee "$device_mixing_output"
print(f"Written to {device_mixing_output}")


## 6. Stage B — M3-5: full block, `stride=5`, synthetic weights, 6 seeds

Same 6 seeds M1's M3-5 sweep used (`m3-block-results.md`): `57305` (this case's own default
synthetic-weight seed), then `1, 2, 3, 42, 12345`. Each seed's full stdout (per-layer
divergence report, PRIMARY GATE, RESULT) is appended to one file, matching how the M1 run's
6-seed sweep is reported as one block in the results doc.


In [ ]:
import os
if not os.path.isdir("/content/higgs-local-test/docs/research/mojo-max"):
    print("/content/higgs-local-test missing -- re-cloning (a Colab runtime hiccup likely wiped /content)")
    !rm -rf /content/higgs-local-test
    !git clone --quiet --depth 1 --branch main https://github.com/vedmalex/higgs-local-test.git /content/higgs-local-test
    assert os.path.isdir("/content/higgs-local-test/docs/research/mojo-max"), "re-clone still failed -- check network/repo access"

SEEDS_M35 = [57305, 1, 2, 3, 42, 12345]
m35_output = OUTPUT_DIR / "m3-block-m35-output-t4.txt"
chunks = []
for seed in SEEDS_M35:
    print(f"=== M3-5 stride=5 synthetic seed={seed} ===")
    proc = !cd /content/mojo-probe-t4 && pixi run python -u /content/higgs-local-test/docs/research/mojo-max/m3_decoder_block_prototype.py --stride 5 --seed {seed}
    text = "\n".join(proc)
    print(text)
    chunks.append(f"=== M3-5 stride=5 synthetic seed={seed} ===\n{text}\n")
m35_output.write_text("\n".join(chunks), encoding="utf-8")
print(f"Written to {m35_output}")


## 7. Stage C — real-weight extraction environment (`.venv-tts`)

`m3_decoder_block_prototype.py --real-weights` needs the real stride-5 block-1 weights as a
plain-FP32 `.npz` cache (`docs/research/mojo-max/.m3_real_weights_block1.npz`, gitignored,
regeneratable). Reading the checkpoint's raw BF16 tensors needs `torch`+`transformers`+
`safetensors`, which cannot live in the pixi/MAX env (confirmed on M1: `pixi run python -c
"import torch"` → `ModuleNotFoundError`). This mirrors the M1 run's `.venv-tts` split exactly
(`m3-block-results.md`'s M3-6 section: "torch 2.13.0, transformers 5.15.1, safetensors 0.8.0").

**Open question for whoever runs this**: `bosonai/higgs-tts-3-4b` was already sitting in this
repo's local HF cache on the M1 host from prior benchmark work — on a fresh Colab VM it is NOT
cached and must be downloaded here. The checkpoint is a 3-4B-parameter model (927 tensors
total, BF16); this download's size/time on Colab's network was not measured before writing
this notebook. If disk or time becomes a problem, only `model.safetensors.index.json` plus the
one shard holding `acoustic_decoder.block.1.*` is strictly needed — this cell downloads
everything matching `*.safetensors`/`*.json` for simplicity, not for the minimum download.

**Drive-first, per the user's request**: this cell now checks `MyDrive/higgs-benchmark/model-cache/higgs-tts-3-4b.tar` (built by `notebooks/model_prefetch_to_drive.ipynb`) before touching the network, and extracts it directly into the default HF cache if present -- only falling back to `snapshot_download` if that archive is missing from Drive.


In [ ]:
# Colab's stock Python venv module has a broken ensurepip on some images ("No module
# named pip" right after venv creation) -- virtualenv bundles its own pip wheel and does
# not depend on ensurepip at all, so it sidesteps that failure mode entirely.
!pip install --quiet virtualenv
!python3 -m virtualenv --quiet /content/higgs-local-test/.venv-tts
!/content/higgs-local-test/.venv-tts/bin/python -m pip install --quiet --upgrade pip setuptools wheel
# Pinned to the exact versions the M1 run used; if any of these three has no wheel for this
# Colab image, drop the pin (unpinned install as a fallback) rather than blocking the whole run.
!/content/higgs-local-test/.venv-tts/bin/python -m pip install --quiet \
    "torch==2.13.0" "transformers==5.15.1" "safetensors==0.8.0" huggingface_hub numpy \
    || /content/higgs-local-test/.venv-tts/bin/python -m pip install --quiet \
    torch transformers safetensors huggingface_hub numpy
!/content/higgs-local-test/.venv-tts/bin/python -c "import torch, transformers, safetensors; print('torch', torch.__version__, 'transformers', transformers.__version__, 'safetensors', safetensors.__version__)"


In [ ]:
# Prefer the Drive backup (built by notebooks/model_prefetch_to_drive.ipynb) over a fresh
# network download -- the checkpoint (927 tensors, BF16, 3-4B params) was never measured on
# Colab's network before this notebook was written, and the archive already exists on Drive
# with the exact layout snapshot_download produces (packed with arcname=model_dir.name, i.e.
# "models--bosonai--higgs-tts-3-4b"), so extracting it reproduces the default HF cache exactly.
import subprocess
import tarfile
from pathlib import Path

DRIVE_MODEL_ARCHIVE = Path("/content/drive/MyDrive/higgs-benchmark/model-cache/higgs-tts-3-4b.tar")
HF_HUB_DIR = Path.home() / ".cache" / "huggingface" / "hub"

if USE_DRIVE and DRIVE_MODEL_ARCHIVE.is_file():
    print(f"Found on Drive: {DRIVE_MODEL_ARCHIVE} ({DRIVE_MODEL_ARCHIVE.stat().st_size / 1e9:.2f} GB) -- extracting, no download needed")
    HF_HUB_DIR.mkdir(parents=True, exist_ok=True)
    with tarfile.open(DRIVE_MODEL_ARCHIVE, "r") as tf:
        tf.extractall(HF_HUB_DIR)
    extracted = HF_HUB_DIR / "models--bosonai--higgs-tts-3-4b"
    assert extracted.is_dir(), f"expected extracted cache dir missing: {extracted}"
    print(f"extracted to {extracted}")
else:
    if USE_DRIVE:
        print(f"Not found on Drive at {DRIVE_MODEL_ARCHIVE} -- falling back to snapshot_download")
    else:
        print("USE_DRIVE is False -- downloading fresh")
    # Download the real checkpoint into the DEFAULT HF cache ($HOME/.cache/huggingface/hub) --
    # m3_block_weights.py's find_snapshot_dir() looks there specifically, with no override.
    # subprocess.run (not a `!` shell magic) so this stays unambiguous inside this if/else block.
    download_script = (
        "from huggingface_hub import snapshot_download\n"
        "path = snapshot_download(repo_id='bosonai/higgs-tts-3-4b', "
        "allow_patterns=['*.json', '*.safetensors'])\n"
        "print('snapshot at', path)\n"
    )
    result = subprocess.run(
        ["/content/higgs-local-test/.venv-tts/bin/python", "-c", download_script],
        check=True,
    )


In [ ]:
# Explicit, eager run (rather than relying on m3_decoder_block_prototype.py's own lazy
# auto-invoke of this exact script) so any environment/checkpoint problem surfaces here, with a
# clear traceback, instead of buried inside a later subprocess's captured stdout/stderr.
!/content/higgs-local-test/.venv-tts/bin/python /content/higgs-local-test/docs/research/mojo-max/m3_real_weights_export.py --seed 99 --seq-len 20


## 8. Stage D — M3-6: full block, `stride=5`, REAL weights, seed 99

Seed 99 matches M1's M3-6 run (`m3-block-results.md`: "seed=99, the default — matches M3-4's
real-weight cross-check convention"). `--real-weights` reads the `.npz` cache Stage C just
produced with NumPy alone — no `torch` import happens inside the pixi/MAX env.


In [ ]:
import os
if not os.path.isdir("/content/higgs-local-test/docs/research/mojo-max"):
    print("/content/higgs-local-test missing -- re-cloning (a Colab runtime hiccup likely wiped /content)")
    !rm -rf /content/higgs-local-test
    !git clone --quiet --depth 1 --branch main https://github.com/vedmalex/higgs-local-test.git /content/higgs-local-test
    assert os.path.isdir("/content/higgs-local-test/docs/research/mojo-max"), "re-clone still failed -- check network/repo access"

m36_output = OUTPUT_DIR / "m3-block-m36-output-t4.txt"
!cd /content/mojo-probe-t4 && pixi run python -u /content/higgs-local-test/docs/research/mojo-max/m3_decoder_block_prototype.py --real-weights --stride 5 --seed 99 2>&1 | tee "$m36_output"
print(f"Written to {m36_output}")


## 9. Stage E — M3-7: full block, `stride=8`, synthetic weights, 6 seeds

Same 6 seeds M1's M3-7 sweep used: `24601` (this case's own default synthetic-weight seed),
then `1, 2, 3, 42, 12345` — the same trailing 5 seeds M3-5 also swept, per `m3-block-results.md`.
`--real-weights` is rejected by the script for any stride other than 5, so this is
synthetic-only, matching M1.


In [ ]:
import os
if not os.path.isdir("/content/higgs-local-test/docs/research/mojo-max"):
    print("/content/higgs-local-test missing -- re-cloning (a Colab runtime hiccup likely wiped /content)")
    !rm -rf /content/higgs-local-test
    !git clone --quiet --depth 1 --branch main https://github.com/vedmalex/higgs-local-test.git /content/higgs-local-test
    assert os.path.isdir("/content/higgs-local-test/docs/research/mojo-max"), "re-clone still failed -- check network/repo access"

SEEDS_M37 = [24601, 1, 2, 3, 42, 12345]
m37_output = OUTPUT_DIR / "m3-block-m37-output-t4.txt"
chunks = []
for seed in SEEDS_M37:
    print(f"=== M3-7 stride=8 synthetic seed={seed} ===")
    proc = !cd /content/mojo-probe-t4 && pixi run python -u /content/higgs-local-test/docs/research/mojo-max/m3_decoder_block_prototype.py --stride 8 --seed {seed}
    text = "\n".join(proc)
    print(text)
    chunks.append(f"=== M3-7 stride=8 synthetic seed={seed} ===\n{text}\n")
m37_output.write_text("\n".join(chunks), encoding="utf-8")
print(f"Written to {m37_output}")


## 10. Stage F — M3-9: BF16-storage precision pass, real weights, `stride=5`, seed 99

Three variants over the same real block-1 weights/input Stage C exported, per M1's M3-9 run:
`fp32` (regression check against M3-6), `bf16-cast` (BF16 storage, explicit FP32 compute, cast
back to BF16 only at the block-output boundary), `bf16-nocast` (BF16 storage AND compute,
no explicit cast anywhere). On M1/Metal these landed `fine` / `breaks` / `breaks` respectively
(`m3-block-results.md`'s M3-9 section) — per `m1-responsibility-map.md` §10, any "BF16 is safe"
conclusion is Metal-only until this T4 run confirms or contradicts it, since Tesla T4/Turing
has no BF16 tensor cores and M0 already flagged that a T4 "PASS" here could be MAX
transparently falling back to another path rather than genuine hardware BF16 execution — this
run's numbers should be read with that caveat in mind, not read as automatically settling it.


In [ ]:
import os
if not os.path.isdir("/content/higgs-local-test/docs/research/mojo-max"):
    print("/content/higgs-local-test missing -- re-cloning (a Colab runtime hiccup likely wiped /content)")
    !rm -rf /content/higgs-local-test
    !git clone --quiet --depth 1 --branch main https://github.com/vedmalex/higgs-local-test.git /content/higgs-local-test
    assert os.path.isdir("/content/higgs-local-test/docs/research/mojo-max"), "re-clone still failed -- check network/repo access"

PRECISIONS_M39 = ["fp32", "bf16-cast", "bf16-nocast"]
m39_output = OUTPUT_DIR / "m3-block-m39-output-t4.txt"
chunks = []
for precision in PRECISIONS_M39:
    print(f"=== M3-9 stride=5 real-weights precision={precision} seed=99 ===")
    proc = !cd /content/mojo-probe-t4 && pixi run python -u /content/higgs-local-test/docs/research/mojo-max/m3_decoder_block_prototype.py --real-weights --stride 5 --precision {precision}
    text = "\n".join(proc)
    print(text)
    chunks.append(f"=== M3-9 stride=5 real-weights precision={precision} seed=99 ===\n{text}\n")
m39_output.write_text("\n".join(chunks), encoding="utf-8")
print(f"Written to {m39_output}")


## 11. Assemble the plan's canonical combined file

`m3-plan.md`'s M3-10 done-criterion names one file, `m3-block-output-t4.txt` (singular) — the
per-stage files above are kept too (their names correspond 1:1 to the per-task sections in
`m3-block-results.md` that will eventually cite them), but this cell also concatenates the four
`m3_decoder_block_prototype.py` runs (M3-5/M3-6/M3-7/M3-9 — NOT M3-1, which gets its own
device-mixing-specific file per the plan's own note) into that one canonical file.


In [ ]:
combined_output = OUTPUT_DIR / "m3-block-output-t4.txt"
parts = []
for name in ["m3-block-m35-output-t4.txt", "m3-block-m36-output-t4.txt",
             "m3-block-m37-output-t4.txt", "m3-block-m39-output-t4.txt"]:
    p = OUTPUT_DIR / name
    parts.append(f"########## {name} ##########\n{p.read_text(encoding='utf-8')}\n")
combined_output.write_text("\n".join(parts), encoding="utf-8")
print(f"Written to {combined_output}")


## 12. Завершение: сброс на Диск, затем безусловное отключение

Все результаты уже лежат на Диске (`OUTPUT_DIR`) — каждый записан сразу после своей ячейки, а
не в самом конце, поэтому обрыв на середине прогона не теряет то, что уже готово. Здесь только
перечисление того, что сохранено, и безусловное отключение ВМ.


In [ ]:
try:
    print(f"На Диске сохранено: {OUTPUT_DIR}")
    for artefact in sorted(OUTPUT_DIR.iterdir()):
        if artefact.is_file() and not artefact.name.startswith("."):
            print(f"  - {artefact.relative_to(WORKSPACE)} ({artefact.stat().st_size / 1024:.1f} KB)")
except Exception as error:
    print(f"не удалось перечислить артефакты: {error!r}")

try:
    if USE_DRIVE:
        from google.colab import drive
        drive.flush_and_unmount()
        print("💾 Данные синхронизированы: MyDrive/higgs-benchmark/mojo-max-m3/")
finally:
    # Безусловно: квота Colab не должна гореть на простое. Разбор результатов --
    # по файлам на Диске, а не по выводу ячеек.
    from google.colab import runtime
    print("🛑 Отключение ВМ...")
    runtime.unassign()
